In [ ]:
!pip install tensorflow kaggle
import os
import string
import pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical, Sequence
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add

from google.colab import drive
drive.mount('/content/drive')

import getpass
os.environ['KAGGLE_API_TOKEN'] = getpass.getpass('Paste your KAGGLE_API_TOKEN: ')

!kaggle datasets download -d adityajn105/flickr8k --unzip

# Set output to Google Drive so nothing is lost if disconnected!
!mkdir -p /content/drive/MyDrive/ImageWeaver_Models
DRIVE_DIR = '/content/drive/MyDrive/ImageWeaver_Models'
BASE_DIR = '/content'

IMAGE_DIR = os.path.join(BASE_DIR, 'Images')
CAPTION_FILE = os.path.join(BASE_DIR, 'captions.txt')
FEATURES_FILE = os.path.join(DRIVE_DIR, 'features.pkl')
TOKENIZER_FILE = os.path.join(DRIVE_DIR, 'tokenizer.pkl')
MODEL_FILE = os.path.join(DRIVE_DIR, 'model.h5')


In [ ]:
def define_model(vocab_size, max_length):
    inputs1 = Input(shape=(2048,))
    fe1 = Dropout(0.5)(inputs1)
    fe2 = Dense(256, activation='relu')(fe1)
    inputs2 = Input(shape=(max_length,))
    se1 = Embedding(vocab_size, 256, mask_zero=False)(inputs2)
    se2 = Dropout(0.5)(se1)
    se3 = LSTM(256)(se2)
    decoder1 = add([fe2, se3])
    decoder2 = Dense(256, activation='relu')(decoder1)
    outputs = Dense(vocab_size, activation='softmax')(decoder2)
    model = Model(inputs=[inputs1, inputs2], outputs=outputs)
    model.compile(loss='categorical_crossentropy', optimizer='adam')
    return model

def extract_features(directory):
    model = InceptionV3()
    model = Model(inputs=model.inputs, outputs=model.layers[-2].output)
    features = dict()
    img_list = os.listdir(directory)
    print(f'Found {len(img_list)} images')
    for idx, name in enumerate(img_list):
        filename = os.path.join(directory, name)
        image = load_img(filename, target_size=(299, 299))
        image = img_to_array(image)
        image = image.reshape((1, image.shape[0], image.shape[1], image.shape[2]))
        image = preprocess_input(image)
        feature = model.predict(image, verbose=0)
        image_id = name.split('.')[0]
        features[image_id] = feature
        if idx % 500 == 0: print(f'> {idx}')
    return features

def load_doc(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        text = f.read()
    return text

def load_descriptions(doc, dataset):
    mapping = dict()
    for line in doc.split('\n'):
        tokens = line.split(',')
        if len(tokens) < 2: continue
        if tokens[0] == 'image': continue
        image_id, image_desc = tokens[0], tokens[1:]
        image_id = image_id.split('.')[0]
        image_desc = ' '.join(image_desc)
        if image_id in dataset:
            if image_id not in mapping: mapping[image_id] = list()
            mapping[image_id].append(image_desc)
    return mapping

def clean_descriptions(descriptions):
    table = str.maketrans('', '', string.punctuation)
    for key, desc_list in descriptions.items():
        for i in range(len(desc_list)):
            desc = desc_list[i].split()
            desc = [word.lower() for word in desc]
            desc = [w.translate(table) for w in desc]
            desc = [word for word in desc if len(word)>1]
            desc = [word for word in desc if word.isalpha()]
            desc_list[i] = 'startseq ' + ' '.join(desc) + ' endseq'


In [ ]:
def to_lines(descriptions):
    all_desc = list()
    for key in descriptions.keys():
        [all_desc.append(d) for d in descriptions[key]]
    return all_desc

def max_length(descriptions):
    lines = to_lines(descriptions)
    return max(len(d.split()) for d in lines)

class DataGenerator(Sequence):
    def __init__(self, descriptions, features, tokenizer, max_length, vocab_size, batch_size=32):
        super().__init__()
        self.descriptions = descriptions
        self.features = features
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.vocab_size = vocab_size
        self.batch_size = batch_size
        self.keys = list(descriptions.keys())
    def __len__(self):
        return int(np.ceil(len(self.keys) / float(self.batch_size)))
    def __getitem__(self, idx):
        batch_keys = self.keys[idx * self.batch_size:(idx + 1) * self.batch_size]
        X1, X2, y = list(), list(), list()
        for key in batch_keys:
            desc_list = self.descriptions[key]
            if key not in self.features: continue
            feature = self.features[key][0]
            for desc in desc_list:
                seq = self.tokenizer.texts_to_sequences([desc])[0]
                for i in range(1, len(seq)):
                    in_seq, out_seq = seq[:i], seq[i]
                    in_seq = pad_sequences([in_seq], maxlen=self.max_length)[0]
                    out_seq = to_categorical([out_seq], num_classes=self.vocab_size)[0]
                    X1.append(feature)
                    X2.append(in_seq)
                    y.append(out_seq)
        return (np.array(X1), np.array(X2)), np.array(y)


In [ ]:
if not os.path.exists(FEATURES_FILE):
    print('Starting Feature Extraction... This takes a few minutes.')
    features = extract_features(IMAGE_DIR)
    with open(FEATURES_FILE, 'wb') as f: pickle.dump(features, f)
else:
    print('Found existing features.pkl! Skipping feature extraction.')
    with open(FEATURES_FILE, 'rb') as f: features = pickle.load(f)

doc = load_doc(CAPTION_FILE)
descriptions = load_descriptions(doc, features)
clean_descriptions(descriptions)

lines = to_lines(descriptions)
tokenizer = Tokenizer()
tokenizer.fit_on_texts(lines)
with open(TOKENIZER_FILE, 'wb') as f: pickle.dump(tokenizer, f)

vocab_size = len(tokenizer.word_index) + 1
max_len = max_length(descriptions)

model = define_model(vocab_size, max_len)
generator = DataGenerator(descriptions, features, tokenizer, max_len, vocab_size, batch_size=32)

checkpoint = tf.keras.callbacks.ModelCheckpoint(MODEL_FILE, monitor='loss', verbose=1, save_best_only=True, mode='min')

model.fit(generator, epochs=20, verbose=1, callbacks=[checkpoint])
